### Markus Hoehn and Leonard Collomb

# Wildfire Damage Risk Modeling: Data Extraction and Feature Engineering

This notebook builds the hex-level dataset used in later analysis and modeling. We aggregate structure-level damage inspection records to H3 hexagons, then attach terrain features from a DEM and vegetation and fuel layers from LANDFIRE.

After building the raw hex-level table, we add a small set of H3 neighborhood features so nearby vegetation and fuels can inform the urban hexes we care about most.

## Data sources

- [Palisades damage inspection data](https://data.lacounty.gov/datasets/CALFIRE-Forestry::dins-2025-palisades-public-view/about)
- [Eaton damage inspection data](https://data.lacounty.gov/datasets/CALFIRE-Forestry::dins-2025-eaton-public-view/about)
- [USGS / The National Map fire impact DEM downloads](https://prd-tnm.s3.amazonaws.com/index.html?prefix=CA_FireImpactZone_2025_PRELIMINARY/)
- [WhiteboxTools geomorphometric analysis tools](https://www.whiteboxgeo.com/manual/wbt_book/available_tools/geomorphometric_analysis.html#geomorphometric-analysis)
- [LANDFIRE vegetation](https://landfire.gov/vegetation)
- [LANDFIRE fuel](https://landfire.gov/fuel)

In [1]:
import geopandas as gpd
import pandas as pd

fire_name = "Palisades"  # or "Eaton"

# Convert to EPSG:4326 for h3 indexing
gdf = gpd.read_file(f"{fire_name}_Public_View.geojson").to_crs(4326)
gdf.head()

,OBJECTID,GLOBALID,DAMAGE,STRUCTURETYPE,geometry
0,1,749289c2-99a6-4d08-a870-16a6e499021e,Destroyed (>50%),Single Family Residence Multi Story,POINT (-118.65963 34.03913)
1,2,12de8e53-d11c-40c4-b628-d861d62cc061,Destroyed (>50%),Single Family Residence Multi Story,POINT (-118.65949 34.03909)
2,4,3d521301-742c-4009-8230-cfff2f4b5125,Destroyed (>50%),Multi Family Residence Multi Story,POINT (-118.65761 34.0394)
3,5,11e11323-9a48-41c7-9d22-12ae0f054089,Destroyed (>50%),Single Family Residence Single Story,POINT (-118.65663 34.03881)
4,6,0023c6d1-86f7-44bb-afaa-00e36c6a9538,Destroyed (>50%),Single Family Residence Single Story,POINT (-118.65606 34.03873)


In [2]:
!pip install h3pandas -q

In [3]:
import h3pandas
from shapely.geometry import box

RES = 10

# Simple study-area buffer so the context table includes nearby wildland too.
CONTEXT_BUFFER_DEG = 0.05

damage_cols = [
    "No Damage",
    "Affected (1-9%)",
    "Minor (10-25%)",
    "Major (26-50%)",
    "Destroyed (>50%)",
]

minx, miny, maxx, maxy = gdf.total_bounds

study_area = gpd.GeoDataFrame(
    geometry=[box(
        minx - CONTEXT_BUFFER_DEG,
        miny - CONTEXT_BUFFER_DEG,
        maxx + CONTEXT_BUFFER_DEG,
        maxy + CONTEXT_BUFFER_DEG,
    )],
    crs=4326,
)

context_hexes = (
    study_area.h3.polyfill(RES, explode=True)[["h3_polyfill"]]
    .drop_duplicates()
    .rename(columns={"h3_polyfill": "h3_10"})
    .set_index("h3_10")
)

context_hexes.index.name = "h3_10"
context_hexes = context_hexes.h3.h3_to_geo_boundary()[["geometry"]].sort_index()

context_hexes.head()

,geometry
h3_10,
8a29a1800007fff,"POLYGON ((-118.60884 34.08824, -118.60819 34.0..."
8a29a180000ffff,"POLYGON ((-118.60985 34.08915, -118.6092 34.08..."
8a29a1800017fff,"POLYGON ((-118.60736 34.08853, -118.60671 34.0..."
8a29a180001ffff,"POLYGON ((-118.60837 34.08944, -118.60772 34.0..."
8a29a1800027fff,"POLYGON ((-118.60931 34.08704, -118.60866 34.0..."


In [4]:
damage_hexes = (
    gdf[["geometry"]]
    .join(pd.get_dummies(gdf["DAMAGE"], dtype=int))
    .h3.geo_to_h3_aggregate(RES, return_geometry=False)
    .reindex(columns=damage_cols, fill_value=0)
)

damage_hexes["total_count"] = damage_hexes[damage_cols].sum(axis=1)

# binary target variable
damage_hexes["is_damaged"] = damage_hexes[damage_cols[1:]].sum(axis=1) > 0

damage_hexes.head()

,No Damage,Affected (1-9%),Minor (10-25%),Major (26-50%),Destroyed (>50%),total_count,is_damaged
h3_10,,,,,,,
8a29a1800427fff,2,0,0,0,0,2,False
8a29a1800507fff,3,0,0,0,0,3,False
8a29a180050ffff,1,0,0,0,0,1,False
8a29a1800517fff,3,0,0,0,0,3,False
8a29a180052ffff,4,0,0,0,0,4,False


In [5]:
structure_hexes = (
    gdf[["geometry"]]
    .join(pd.get_dummies(gdf["STRUCTURETYPE"], dtype=int))
    .h3.geo_to_h3_aggregate(RES)
)

structure_cols = [col for col in structure_hexes.columns if col not in ["geometry", "total_count"]]

In [6]:
# Join the labeled building hexes onto the larger context grid.
hexes = (
    context_hexes
    .join(damage_hexes[damage_cols + ["total_count", "is_damaged"]], how="left")
    .join(structure_hexes[structure_cols], how="left")
)

hexes[damage_cols + ["total_count"]] = hexes[damage_cols + ["total_count"]].fillna(0)
hexes["is_damaged"] = hexes["is_damaged"].fillna(False)
hexes[structure_cols] = hexes[structure_cols].fillna(0)

## DEM data

We next add topographic features from a digital elevation model (DEM). A DEM is a raster in which each cell stores the estimated elevation of the bare-earth ground surface. The DEM is downloaded as tiles, merged into one raster, resampled to a working resolution, and then used to derive terrain features with WhiteboxTools.

In [7]:
import os
import requests

dem_dir = os.path.abspath(f"{fire_name}_dem_tiles")
dem_work_dir = os.path.abspath(f"{fire_name}_dem_work")

os.makedirs(dem_dir, exist_ok=True)
os.makedirs(dem_work_dir, exist_ok=True)

dem_manifest_url = (
    f"https://prd-tnm.s3.amazonaws.com/"
    f"CA_FireImpactZone_2025_PRELIMINARY/{fire_name}/"
    f"bare_earth/be_rasters/0_file_download_links.txt"
)

# Work at 20 m to keep compute manageable while staying much finer than the H3 hexes.
target_dem_res = 20

dem_manifest_text = requests.get(dem_manifest_url).text
dem_urls = [
    line.strip()
    for line in dem_manifest_text.splitlines()
    if line.strip().endswith(".tif")
]

len(dem_urls), dem_urls[:3]

(534,
 ['https://prd-tnm.s3.us-west-2.amazonaws.com/CA_FireImpactZone_2025_PRELIMINARY/Palisades/bare_earth/be_rasters/Prelim_11SLT003420037710.tif',
  'https://prd-tnm.s3.us-west-2.amazonaws.com/CA_FireImpactZone_2025_PRELIMINARY/Palisades/bare_earth/be_rasters/Prelim_11SLT003420037717.tif',
  'https://prd-tnm.s3.us-west-2.amazonaws.com/CA_FireImpactZone_2025_PRELIMINARY/Palisades/bare_earth/be_rasters/Prelim_11SLT003420037725.tif'])

In [8]:
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.merge import merge

for url in dem_urls:
    out_path = os.path.join(dem_dir, os.path.basename(url))
    if not os.path.exists(out_path):
        r = requests.get(url)
        r.raise_for_status()
        with open(out_path, "wb") as f:
            f.write(r.content)

dem_tile_paths = sorted(
    os.path.join(dem_dir, name)
    for name in os.listdir(dem_dir)
    if name.endswith(".tif")
)

srcs = [rasterio.open(path) for path in dem_tile_paths]
merged_arr, merged_transform = merge(srcs)

big_dem_path = os.path.join(dem_work_dir, f"{fire_name}_big_dem.tif")

big_profile = srcs[0].profile | {
    "height": merged_arr.shape[1],
    "width": merged_arr.shape[2],
    "transform": merged_transform,
    "count": 1,
}

with rasterio.open(big_dem_path, "w", **big_profile) as dst:
    dst.write(merged_arr)

for src in srcs:
    src.close()

resampled_dem_path = os.path.join(
    dem_work_dir,
    f"{fire_name}_dem_resampled_{target_dem_res}m.tif"
)

with rasterio.open(big_dem_path) as src:
    xres, yres = map(abs, src.res)

    out_height = int(round(src.height * yres / target_dem_res))
    out_width = int(round(src.width * xres / target_dem_res))

    dem = src.read(
        1,
        out_shape=(out_height, out_width),
        resampling=Resampling.bilinear,
        masked=True,
    )

    transform = src.transform * src.transform.scale(
        src.width / out_width,
        src.height / out_height,
    )

    nodata = src.nodata if src.nodata is not None else -9999.0

    out_profile = src.profile | {
        "height": out_height,
        "width": out_width,
        "transform": transform,
        "count": 1,
        "dtype": "float32",
        "nodata": nodata,
    }

    with rasterio.open(resampled_dem_path, "w", **out_profile) as dst:
        dst.write(dem.filled(nodata), 1)

resampled_dem_path

'/content/Palisades_dem_work/Palisades_dem_resampled_20m.tif'

## Terrain features

Using WhiteboxTools, we derive several terrain features from the DEM:

- **Slope**: how steep the ground is, measured in degrees.
- **Aspect**: which direction the slope faces, measured clockwise from north.
- **TRI (terrain ruggedness index)**: how rough the terrain is locally, based on how different a cell is from its neighbors.
- **Plan curvature**: how the land surface curves sideways across a slope; this helps describe whether flow tends to spread out or converge.
- **Profile curvature**: how the land surface curves in the downhill direction; this helps describe whether flow tends to speed up or slow down.
- **TPI (topographic position index)**: whether a location is high or low relative to the surrounding terrain:
  
  $$
  \mathrm{TPI} = z_0 - \bar{z}_{\text{neighborhood}}
  $$
  
  where $z_0$ is the elevation of the focal cell. Positive values indicate locally elevated areas such as ridges, while negative values indicate lower areas such as valleys.
- **TWI (topographic wetness index)**: a simple index of how likely water is to accumulate at a location:
  
  $$
  \mathrm{TWI} = \ln\left(\frac{A_s}{\tan(\beta)}\right)
  $$
  
  where $A_s$ is specific contributing area and $\beta$ is slope. Higher values generally correspond to places that receive more upslope flow and are more likely to stay wetter.

We compute TPI at two window sizes to capture both local and broader landform context.

In [9]:
!pip install -U whitebox -q

In [10]:
from whitebox.whitebox_tools import WhiteboxTools

wbt = WhiteboxTools()
wbt.set_verbose_mode(False)
wbt.set_working_dir(dem_work_dir)

analysis_dem_path = resampled_dem_path
analysis_dem_file = os.path.basename(analysis_dem_path)

breached_dem_file = "breached_dem.tif"
slope_file = f"{fire_name}_slope_deg.tif"
aspect_file = f"{fire_name}_aspect_deg.tif"
tri_file = f"{fire_name}_tri.tif"
plan_file = f"{fire_name}_plan_curvature.tif"
profile_file = f"{fire_name}_profile_curvature.tif"
tpi_small_file = f"{fire_name}_tpi_small.tif"
tpi_large_file = f"{fire_name}_tpi_large.tif"
sca_file = f"{fire_name}_sca.tif"
twi_file = f"{fire_name}_twi.tif"

wbt.slope(dem=analysis_dem_file, output=slope_file, units="degrees")
wbt.aspect(dem=analysis_dem_file, output=aspect_file)
wbt.ruggedness_index(dem=analysis_dem_file, output=tri_file)
wbt.plan_curvature(dem=analysis_dem_file, output=plan_file)
wbt.profile_curvature(dem=analysis_dem_file, output=profile_file)

wbt.diff_from_mean_elev(dem=analysis_dem_file, output=tpi_small_file, filterx=11, filtery=11)
wbt.diff_from_mean_elev(dem=analysis_dem_file, output=tpi_large_file, filterx=31, filtery=31)

wbt.breach_depressions(dem=analysis_dem_file, output=breached_dem_file)

wbt.d8_flow_accumulation(
    i=breached_dem_file,
    output=sca_file,
    out_type="specific contributing area",
)

wbt.wetness_index(sca=sca_file, slope=slope_file, output=twi_file)

dem_raster_paths = {
    "elevation": analysis_dem_path,
    "slope_deg": os.path.join(dem_work_dir, slope_file),
    "aspect_deg": os.path.join(dem_work_dir, aspect_file),
    "tri": os.path.join(dem_work_dir, tri_file),
    "plan_curvature": os.path.join(dem_work_dir, plan_file),
    "profile_curvature": os.path.join(dem_work_dir, profile_file),
    "tpi_small": os.path.join(dem_work_dir, tpi_small_file),
    "tpi_large": os.path.join(dem_work_dir, tpi_large_file),
    "twi": os.path.join(dem_work_dir, twi_file),
}

dem_raster_paths

{'elevation': '/content/Palisades_dem_work/Palisades_dem_resampled_20m.tif',
 'slope_deg': '/content/Palisades_dem_work/Palisades_slope_deg.tif',
 'aspect_deg': '/content/Palisades_dem_work/Palisades_aspect_deg.tif',
 'tri': '/content/Palisades_dem_work/Palisades_tri.tif',
 'plan_curvature': '/content/Palisades_dem_work/Palisades_plan_curvature.tif',
 'profile_curvature': '/content/Palisades_dem_work/Palisades_profile_curvature.tif',
 'tpi_small': '/content/Palisades_dem_work/Palisades_tpi_small.tif',
 'tpi_large': '/content/Palisades_dem_work/Palisades_tpi_large.tif',
 'twi': '/content/Palisades_dem_work/Palisades_twi.tif'}

In [11]:
with rasterio.open(dem_raster_paths["elevation"]) as src:
    elevation = src.read(1, masked=True)
    rows, cols = np.where(~elevation.mask)
    xs, ys = rasterio.transform.xy(src.transform, rows, cols, offset="center")
    crs = src.crs

dem_points = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(xs, ys),
    crs=crs,
)

for feature_name, raster_path in dem_raster_paths.items():
    with rasterio.open(raster_path) as src:
        dem_points[feature_name] = src.read(1, masked=True)[rows, cols].filled(np.nan)

# Use northness / eastness instead of raw aspect so direction is encoded continuously.
dem_points["northness"] = np.cos(np.deg2rad(dem_points["aspect_deg"]))
dem_points["eastness"] = np.sin(np.deg2rad(dem_points["aspect_deg"]))

dem_points = dem_points.drop(columns="aspect_deg").to_crs(4326)
dem_points["dem_raster_count"] = 1

terrain_cols = [
    "elevation",
    "slope_deg",
    "tri",
    "plan_curvature",
    "profile_curvature",
    "tpi_small",
    "tpi_large",
    "twi",
    "northness",
    "eastness",
]

# Mean is a natural hex-level summary for continuous terrain features.
dem_hexes = (
    dem_points[["geometry", "dem_raster_count"] + terrain_cols]
    .h3.geo_to_h3_aggregate(
        RES,
        operation={"dem_raster_count": "sum"} | {col: "mean" for col in terrain_cols},
        return_geometry=False,
    )
)

hexes = hexes.join(dem_hexes, how="left")
hexes.head()

,geometry,No Damage,Affected (1-9%),Minor (10-25%),Major (26-50%),Destroyed (>50%),total_count,is_damaged,Church,Commercial Building Multi Story,...,elevation,slope_deg,tri,plan_curvature,profile_curvature,tpi_small,tpi_large,twi,northness,eastness
h3_10,,,,,,,,,,,,,,,,,,,,,
8a29a1800007fff,"POLYGON ((-118.60884 34.08824, -118.60819 34.0...",0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,...,300.571442,17.553146,5.597445,0.006245,-0.000515,1.513902,3.838872,5.160396,-0.149223,0.883417
8a29a180000ffff,"POLYGON ((-118.60985 34.08915, -118.6092 34.08...",0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,...,300.936523,14.491197,4.723875,0.004537,0.000076,1.842425,7.579220,5.641702,0.896738,0.254519
8a29a1800017fff,"POLYGON ((-118.60736 34.08853, -118.60671 34.0...",0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,...,280.930145,12.462770,4.045117,0.011273,0.002002,6.146007,6.335433,5.144251,-0.215698,0.615917
8a29a180001ffff,"POLYGON ((-118.60837 34.08944, -118.60772 34.0...",0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,...,281.219177,12.143764,3.777644,-0.001544,-0.000397,0.032841,4.661808,5.813115,0.875787,0.432024
8a29a1800027fff,"POLYGON ((-118.60931 34.08704, -118.60866 34.0...",0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,...,311.716583,28.739227,9.857190,-0.000549,-0.001391,-3.235453,-7.394793,5.089401,-0.080159,0.985417


## LANDFIRE layers

Finally, we attach LANDFIRE vegetation and fuel layers. The main layers we use are EVT (Existing Vegetation Type), EVC (Existing Vegetation Cover), EVH (Existing Vegetation Height), and FBFM40 (the 40 Scott and Burgan Fire Behavior Fuel Models).

LANDFIRE's **Existing Vegetation Cover (EVC)** represents the vertically projected percent cover of the live canopy layer within a 30 m cell; informally, it measures how dense the live vegetation is.

LANDFIRE's **Existing Vegetation Type (EVT)** represents the current distribution of ecological vegetation systems. Informally, EVT describes what kind of vegetation system is present.

LANDFIRE's **Existing Vegetation Height (EVH)** represents the average height of the dominant vegetation within a 30 m cell.

LANDFIRE's **40 Scott and Burgan Fire Behavior Fuel Model (FBFM40)** classifies the surface fuel structure that most affects expected fire behavior.

In [12]:
from zipfile import ZipFile
from rasterio.windows import Window, from_bounds
from rasterio.warp import transform_bounds

landfire_dir = f"{fire_name}_landfire"
os.makedirs(landfire_dir, exist_ok=True)

LANDFIRE_URLS = {
    "evt": "https://www.landfire.gov/data-downloads/CONUS_LF2024/LF2024_EVT_CONUS.zip",
    "evc": "https://www.landfire.gov/data-downloads/CONUS_LF2024/LF2024_EVC_CONUS.zip",
    "evh": "https://www.landfire.gov/data-downloads/CONUS_LF2024/LF2024_EVH_CONUS.zip",
    "fbfm40": "https://www.landfire.gov/data-downloads/CONUS_LF2024/LF2024_FBFM40_CONUS.zip",
}

minx, miny, maxx, maxy = gdf.total_bounds
landfire_legends = {}

def aggregate_landfire_mode(url, value_col):
    stem = os.path.splitext(os.path.basename(url))[0]
    zip_path = os.path.join(landfire_dir, f"{stem}.zip")
    extract_dir = os.path.join(landfire_dir, stem)

    if not os.path.exists(zip_path):
        with open(zip_path, "wb") as f:
            f.write(requests.get(url).content)

    if not os.path.isdir(extract_dir):
        os.makedirs(extract_dir, exist_ok=True)
        with ZipFile(zip_path) as zf:
            zf.extractall(extract_dir)

    tif_path = next(
        os.path.join(root, file)
        for root, _, files in os.walk(extract_dir)
        for file in files
        if file.lower().endswith(".tif")
    )
    csv_path = next(
        os.path.join(root, file)
        for root, _, files in os.walk(extract_dir)
        for file in files
        if file.lower().endswith(".csv") and "CSV_Data" in root.split(os.sep)
    )

    with rasterio.open(tif_path) as src:
        window = from_bounds(
            *transform_bounds("EPSG:4326", src.crs, minx, miny, maxx, maxy, densify_pts=21),
            transform=src.transform,
        ).intersection(Window(0, 0, src.width, src.height))

        arr = src.read(1, window=window, masked=True)
        rows, cols = np.where(~arr.mask)
        xs, ys = rasterio.transform.xy(src.window_transform(window), rows, cols, offset="center")
        crs = src.crs

    landfire_points = gpd.GeoDataFrame(
        {value_col: arr.data[rows, cols]},
        geometry=gpd.points_from_xy(xs, ys),
        crs=crs,
    ).to_crs(4326)

    landfire_hexes = landfire_points.h3.geo_to_h3_aggregate(
        RES,
        # These are categorical raster layers, so mode is the natural hex summary.
        operation={value_col: lambda s: s.mode().iloc[0]},
        return_geometry=False,
    )

    legend = pd.read_csv(csv_path)
    legend.columns = legend.columns.str.upper().str.strip()

    return landfire_hexes, legend

for layer_name, url in LANDFIRE_URLS.items():
    value_col = f"{layer_name}_value"
    landfire_hexes, legend = aggregate_landfire_mode(url, value_col)
    hexes = hexes.join(landfire_hexes[[value_col]], how="left")
    landfire_legends[layer_name] = legend

hexes.head()

,geometry,No Damage,Affected (1-9%),Minor (10-25%),Major (26-50%),Destroyed (>50%),total_count,is_damaged,Church,Commercial Building Multi Story,...,profile_curvature,tpi_small,tpi_large,twi,northness,eastness,evt_value,evc_value,evh_value,fbfm40_value
h3_10,,,,,,,,,,,,,,,,,,,,,
8a29a1800007fff,"POLYGON ((-118.60884 34.08824, -118.60819 34.0...",0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,...,-0.000515,1.513902,3.838872,5.160396,-0.149223,0.883417,7092.0,347.0,304.0,122.0
8a29a180000ffff,"POLYGON ((-118.60985 34.08915, -118.6092 34.08...",0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,...,0.000076,1.842425,7.579220,5.641702,0.896738,0.254519,7113.0,153.0,304.0,123.0
8a29a1800017fff,"POLYGON ((-118.60736 34.08853, -118.60671 34.0...",0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,...,0.002002,6.146007,6.335433,5.144251,-0.215698,0.615917,7299.0,25.0,25.0,91.0
8a29a180001ffff,"POLYGON ((-118.60837 34.08944, -118.60772 34.0...",0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,...,-0.000397,0.032841,4.661808,5.813115,0.875787,0.432024,7129.0,17.0,304.0,102.0
8a29a1800027fff,"POLYGON ((-118.60931 34.08704, -118.60866 34.0...",0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,...,-0.001391,-3.235453,-7.394793,5.089401,-0.080159,0.985417,7092.0,22.0,303.0,102.0


In [13]:
evt_legend = landfire_legends["evt"].set_index(
    pd.to_numeric(landfire_legends["evt"]["VALUE"])
)

hexes["evt_lifeform"] = (
    pd.to_numeric(hexes["evt_value"])
    .map(evt_legend["EVT_LF"])
)
hexes["evt_group_name"] = (
    pd.to_numeric(hexes["evt_value"])
    .map(evt_legend["EVT_GP_N"])
)
hexes["evt_subclass"] = (
    pd.to_numeric(hexes["evt_value"])
    .map(evt_legend["EVT_SBCLS"])
)

# EVC / EVH class labels contain numeric anchors we want for modeling.
for layer_name in ["evc", "evh"]:
    legend = landfire_legends[layer_name]
    lookup = pd.Series(
        legend.iloc[:, 1].values,
        index=pd.to_numeric(legend["VALUE"])
    )
    labels = pd.to_numeric(hexes[f"{layer_name}_value"]).map(lookup)
    hexes[f"{layer_name}_numeric"] = (
        labels
        .str.extract(r"=\s*([0-9]*\.?[0-9]+)", expand=False)
        .astype(float)
        .fillna(0)
    )

fbfm40_lookup = pd.Series(
    landfire_legends["fbfm40"].iloc[:, 1].str.strip().str.upper().values,
    index=pd.to_numeric(landfire_legends["fbfm40"]["VALUE"]),
)

hexes["fuel_code"] = (
    pd.to_numeric(hexes["fbfm40_value"])
    .map(fbfm40_lookup)
)
hexes["fuel_family"] = (
    hexes["fuel_code"]
    .str.extract(r"^([A-Z]+)", expand=False)
)
hexes["is_burnable"] = hexes["fuel_family"].isin(["GR", "GS", "SH", "TU", "TL", "SB"])

fuel_map = {
    "NB1": {"climate_type": "None", "fuel_load": "None", "dominant_fuel": "nonburnable", "expected_spread": "None", "expected_flame": "None"},
    "NB2": {"climate_type": "None", "fuel_load": "None", "dominant_fuel": "nonburnable", "expected_spread": "None", "expected_flame": "None"},
    "NB3": {"climate_type": "None", "fuel_load": "None", "dominant_fuel": "nonburnable", "expected_spread": "None", "expected_flame": "None"},
    "NB8": {"climate_type": "None", "fuel_load": "None", "dominant_fuel": "nonburnable", "expected_spread": "None", "expected_flame": "None"},
    "NB9": {"climate_type": "None", "fuel_load": "None", "dominant_fuel": "nonburnable", "expected_spread": "None", "expected_flame": "None"},

    "GR1": {"climate_type": "dry",   "fuel_load": "low",       "dominant_fuel": "grass",  "expected_spread": "low",      "expected_flame": "low"},
    "GR2": {"climate_type": "dry",   "fuel_load": "low",       "dominant_fuel": "grass",  "expected_spread": "None",     "expected_flame": "None"},
    "GR3": {"climate_type": "humid", "fuel_load": "low",       "dominant_fuel": "grass",  "expected_spread": "None",     "expected_flame": "None"},
    "GR4": {"climate_type": "dry",   "fuel_load": "moderate",  "dominant_fuel": "grass",  "expected_spread": "None",     "expected_flame": "None"},
    "GR5": {"climate_type": "humid", "fuel_load": "low",       "dominant_fuel": "grass",  "expected_spread": "None",     "expected_flame": "None"},
    "GR6": {"climate_type": "humid", "fuel_load": "moderate",  "dominant_fuel": "grass",  "expected_spread": "None",     "expected_flame": "None"},
    "GR7": {"climate_type": "dry",   "fuel_load": "high",      "dominant_fuel": "grass",  "expected_spread": "None",     "expected_flame": "None"},
    "GR8": {"climate_type": "humid", "fuel_load": "high",      "dominant_fuel": "grass",  "expected_spread": "extreme",  "expected_flame": "extreme"},
    "GR9": {"climate_type": "humid", "fuel_load": "very_high", "dominant_fuel": "grass",  "expected_spread": "extreme",  "expected_flame": "extreme"},

    "GS1": {"climate_type": "dry",   "fuel_load": "low",       "dominant_fuel": "grass",  "expected_spread": "moderate", "expected_flame": "low"},
    "GS2": {"climate_type": "dry",   "fuel_load": "moderate",  "dominant_fuel": "grass",  "expected_spread": "high",     "expected_flame": "moderate"},
    "GS3": {"climate_type": "humid", "fuel_load": "moderate",  "dominant_fuel": "grass",  "expected_spread": "high",     "expected_flame": "moderate"},
    "GS4": {"climate_type": "humid", "fuel_load": "high",      "dominant_fuel": "grass",  "expected_spread": "high",     "expected_flame": "very_high"},

    "SH1": {"climate_type": "dry",   "fuel_load": "low",       "dominant_fuel": "shrub",  "expected_spread": "low",       "expected_flame": "low"},
    "SH2": {"climate_type": "dry",   "fuel_load": "moderate",  "dominant_fuel": "shrub",  "expected_spread": "low",       "expected_flame": "low"},
    "SH3": {"climate_type": "humid", "fuel_load": "moderate",  "dominant_fuel": "shrub",  "expected_spread": "low",       "expected_flame": "low"},
    "SH4": {"climate_type": "humid", "fuel_load": "low",       "dominant_fuel": "shrub",  "expected_spread": "high",      "expected_flame": "moderate"},
    "SH5": {"climate_type": "dry",   "fuel_load": "high",      "dominant_fuel": "shrub",  "expected_spread": "very_high", "expected_flame": "very_high"},
    "SH6": {"climate_type": "humid", "fuel_load": "low",       "dominant_fuel": "shrub",  "expected_spread": "high",      "expected_flame": "high"},
    "SH7": {"climate_type": "dry",   "fuel_load": "very_high", "dominant_fuel": "shrub",  "expected_spread": "high",      "expected_flame": "very_high"},
    "SH8": {"climate_type": "humid", "fuel_load": "high",      "dominant_fuel": "shrub",  "expected_spread": "high",      "expected_flame": "high"},
    "SH9": {"climate_type": "humid", "fuel_load": "very_high", "dominant_fuel": "shrub",  "expected_spread": "high",      "expected_flame": "high"},

    "TU1": {"climate_type": "dry",   "fuel_load": "low",       "dominant_fuel": "shrub",  "expected_spread": "low",      "expected_flame": "low"},
    "TU2": {"climate_type": "humid", "fuel_load": "moderate",  "dominant_fuel": "shrub",  "expected_spread": "moderate", "expected_flame": "low"},
    "TU3": {"climate_type": "humid", "fuel_load": "moderate",  "dominant_fuel": "shrub",  "expected_spread": "high",     "expected_flame": "moderate"},
    "TU4": {"climate_type": "None",  "fuel_load": "moderate",  "dominant_fuel": "shrub",  "expected_spread": "moderate", "expected_flame": "moderate"},
    "TU5": {"climate_type": "dry",   "fuel_load": "very_high", "dominant_fuel": "shrub",  "expected_spread": "moderate", "expected_flame": "moderate"},

    "TL1": {"climate_type": "None", "fuel_load": "low",       "dominant_fuel": "litter", "expected_spread": "low",      "expected_flame": "low"},
    "TL2": {"climate_type": "None", "fuel_load": "low",       "dominant_fuel": "litter", "expected_spread": "low",      "expected_flame": "low"},
    "TL3": {"climate_type": "None", "fuel_load": "moderate",  "dominant_fuel": "litter", "expected_spread": "low",      "expected_flame": "low"},
    "TL4": {"climate_type": "None", "fuel_load": "moderate",  "dominant_fuel": "litter", "expected_spread": "low",      "expected_flame": "low"},
    "TL5": {"climate_type": "None", "fuel_load": "high",      "dominant_fuel": "litter", "expected_spread": "low",      "expected_flame": "low"},
    "TL6": {"climate_type": "None", "fuel_load": "moderate",  "dominant_fuel": "litter", "expected_spread": "moderate", "expected_flame": "moderate"},
    "TL7": {"climate_type": "None", "fuel_load": "high",      "dominant_fuel": "litter", "expected_spread": "low",      "expected_flame": "low"},
    "TL8": {"climate_type": "None", "fuel_load": "moderate",  "dominant_fuel": "litter", "expected_spread": "moderate", "expected_flame": "low"},
    "TL9": {"climate_type": "None", "fuel_load": "very_high", "dominant_fuel": "litter", "expected_spread": "moderate", "expected_flame": "moderate"},

    "SB1": {"climate_type": "None", "fuel_load": "low",      "dominant_fuel": "slash", "expected_spread": "moderate",  "expected_flame": "low"},
    "SB2": {"climate_type": "None", "fuel_load": "moderate", "dominant_fuel": "slash", "expected_spread": "moderate",  "expected_flame": "low"},
    "SB3": {"climate_type": "None", "fuel_load": "high",     "dominant_fuel": "slash", "expected_spread": "high",      "expected_flame": "high"},
    "SB4": {"climate_type": "None", "fuel_load": "high",     "dominant_fuel": "slash", "expected_spread": "very_high", "expected_flame": "very_high"},
}

for new_col in ["climate_type", "fuel_load", "dominant_fuel", "expected_spread", "expected_flame"]:
    hexes[new_col] = hexes["fuel_code"].map({k: v[new_col] for k, v in fuel_map.items()})

## H3 neighborhood wildfire context

Wildfire risk depends not only on the features inside one H3 hexagon, but also on the surrounding landscape. A hex may be classified as developed even when nearby hexagons are still highly burnable, so we add neighborhood-based features to capture that local context.

We use **k-ring** neighborhoods on the H3 grid, where `k=1` includes the hexagons directly adjacent to a center hex and `k=2` includes hexagons within two steps. From these neighborhoods, we keep the two summaries that were most useful in test modeling:

- **wildfire_ring_numeric**: averages of `evc_numeric`, `evh_numeric`, and `is_burnable` over the `k=1` and `k=2` neighborhoods
- **nearby_mode_cat**: the dominant nearby non-urban category for selected LANDFIRE categorical features over the `k=1` and `k=2` neighborhoods

In [14]:
def make_ring_mean_features(data, value_cols, ring_sizes):
    anchor = pd.DataFrame({"anchor": 1}, index=data.index)
    out_frames = []

    for k in ring_sizes:
        links = anchor.h3.k_ring(k, explode=True).reset_index()
        links = links.rename(columns={
            links.columns[0]: "h3_center",
            "h3_k_ring": "h3_neighbor",
        })
        links = links[links["h3_center"] != links["h3_neighbor"]]

        neighbor_values = data[value_cols].reset_index()
        neighbor_values = neighbor_values.rename(columns={neighbor_values.columns[0]: "h3_neighbor"})

        ring_means = (
            links.merge(neighbor_values, on="h3_neighbor", how="left")
            .groupby("h3_center")[value_cols]
            .mean()
            .add_prefix(f"kring{k}_mean_")
            .reindex(data.index)
        )

        out_frames.append(ring_means)

    return pd.concat(out_frames, axis=1)


def make_ring_mode_features(data, mode_cols, ring_sizes):
    anchor = pd.DataFrame({"anchor": 1}, index=data.index)
    out_frames = []

    for k in ring_sizes:
        links = anchor.h3.k_ring(k, explode=True).reset_index()
        links = links.rename(columns={
            links.columns[0]: "h3_center",
            "h3_k_ring": "h3_neighbor",
        })
        links = links[links["h3_center"] != links["h3_neighbor"]]

        col_frames = []

        for col in mode_cols:
            neighbor_values = data[[col]].reset_index()
            neighbor_values = neighbor_values.rename(columns={neighbor_values.columns[0]: "h3_neighbor"})

            mode_vals = (
                links.merge(neighbor_values, on="h3_neighbor", how="left")
                .dropna(subset=[col])
                .groupby("h3_center")[col]
                .agg(lambda s: s.mode().iloc[0])
                .reindex(data.index)
            )

            col_frames.append(
                mode_vals.to_frame(name=f"kring{k}_nonurban_mode_{col}")
            )

        out_frames.append(pd.concat(col_frames, axis=1))

    return pd.concat(out_frames, axis=1)

In [15]:
hexes["structure_total"] = hexes[structure_cols].sum(axis=1)
hexes["has_any_structure"] = hexes["structure_total"] > 0

structure_prop_cols = [f"{col}_prop" for col in structure_cols]
hexes[structure_prop_cols] = hexes[structure_cols].div(
    hexes["structure_total"].replace(0, 1),
    axis=0,
)

wildfire_ring_numeric = make_ring_mean_features(
    hexes,
    ["evc_numeric", "evh_numeric", "is_burnable"],
    ring_sizes=[1, 2],
)

nearby_mode_source = pd.DataFrame(index=hexes.index)
nearby_mode_source["evt_lifeform"] = hexes["evt_lifeform"].where(
    hexes["evt_lifeform"] != "Developed"
)
nearby_mode_source["evt_group_name"] = hexes["evt_group_name"].where(
    ~hexes["evt_group_name"].str.startswith("Developed", na=False)
    & ~hexes["evt_group_name"].str.contains("Urban", regex=False, na=False)
)
nearby_mode_source["fuel_family"] = hexes["fuel_family"].where(
    hexes["fuel_family"] != "NB"
)
nearby_mode_source["dominant_fuel"] = hexes["dominant_fuel"].where(
    hexes["dominant_fuel"] != "nonburnable"
)
nearby_mode_source["fuel_load"] = hexes["fuel_load"]
nearby_mode_source["expected_spread"] = hexes["expected_spread"]
nearby_mode_source["expected_flame"] = hexes["expected_flame"]

nearby_mode_cat = make_ring_mode_features(
    nearby_mode_source,
    [
        "evt_lifeform",
        "evt_group_name",
        "fuel_family",
        "dominant_fuel",
        "fuel_load",
        "expected_spread",
        "expected_flame",
    ],
    ring_sizes=[1, 2],
)

hexes = hexes.join([
    wildfire_ring_numeric,
    nearby_mode_cat,
])

wildfire_ring_numeric_cols = wildfire_ring_numeric.columns.tolist()
nearby_mode_cat_cols = nearby_mode_cat.columns.tolist()

ring_feature_cols = wildfire_ring_numeric_cols + nearby_mode_cat_cols

hexes[
    wildfire_ring_numeric_cols
    + nearby_mode_cat_cols[:6]
].head()

,kring1_mean_evc_numeric,kring1_mean_evh_numeric,kring1_mean_is_burnable,kring2_mean_evc_numeric,kring2_mean_evh_numeric,kring2_mean_is_burnable,kring1_nonurban_mode_evt_lifeform,kring1_nonurban_mode_evt_group_name,kring1_nonurban_mode_fuel_family,kring1_nonurban_mode_dominant_fuel,kring1_nonurban_mode_fuel_load,kring1_nonurban_mode_expected_spread
h3_10,,,,,,,,,,,,
8a29a1800007fff,16.666667,0.250000,0.666667,14.777778,4.916667,0.555556,Herb,Grassland,GR,grass,low,None
8a29a180000ffff,21.000000,4.866667,0.833333,15.111111,6.138889,0.611111,Herb,Grassland,GR,grass,moderate,None
8a29a1800017fff,7.833333,0.133333,0.333333,12.055556,4.161111,0.444444,Herb,Grassland,GR,grass,None,None
8a29a180001ffff,16.666667,0.133333,0.5,12.388889,3.305556,0.444444,Tree,Western Oak Woodland and Savanna,GS,grass,None,None
8a29a1800027fff,38.000000,9.966667,0.833333,30.000000,5.438889,0.777778,Shrub,California Mixed Evergreen Forest and Woodland,GS,grass,moderate,high


## Save the final modeling tables

We export two CSV files:

- a **context-level** table with every H3 hex in the study area
- a **structure-level** table with only hexes that contain labeled structures

Both outputs include the center-hex features plus the smaller neighborhood feature block used in the final modeling notebook.

In [16]:
hexes["VALUE_evt"] = pd.to_numeric(hexes["evt_value"])
hexes["VALUE_evc"] = pd.to_numeric(hexes["evc_value"])
hexes["VALUE_evh"] = pd.to_numeric(hexes["evh_value"])
hexes["VALUE_fbfm40"] = pd.to_numeric(hexes["fbfm40_value"])

num_cols = (
    structure_cols
    + ["structure_total", "has_any_structure"]
    + structure_prop_cols
    + terrain_cols
    + ["total_count", "evc_numeric", "evh_numeric", "is_burnable"]
)

cat_cols = [
    "evt_lifeform",
    "evt_group_name",
    "evt_subclass",
    "fuel_code",
    "fuel_family",
    "climate_type",
    "fuel_load",
    "dominant_fuel",
    "expected_spread",
    "expected_flame",
]

raw_landfire_cols = [
    "VALUE_evt",
    "VALUE_evc",
    "VALUE_evh",
    "VALUE_fbfm40",
]

final_cols = (
    ["is_damaged"]
    + num_cols
    + cat_cols
    + raw_landfire_cols
    + ring_feature_cols
    + ["geometry"]
)

context_output_path = f"{fire_name}_context_hex_data.csv"
hexes[final_cols].to_csv(context_output_path)

structure_output_path = f"{fire_name}_structure_hex_data.csv"
hexes.loc[hexes["total_count"] > 0, final_cols].to_csv(structure_output_path)

context_output_path, structure_output_path

('Palisades_context_hex_data.csv', 'Palisades_structure_hex_data.csv')